# Experiment 3.1: Hessian-Adaptive Muon 2.0

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/MUON_2_0_hessian_adaptive/muon_2_0_hessian_adaptive.py`

This notebook is the presentation-and-analysis counterpart to the script above. The main experiment is **imported from the script rather than reimplemented here**, so the notebook and script share one execution path.

## Pair-level scope

This is a **48-parameter deep-linear toy problem**:
- 3 layers, each `4 x 4`
- target singular values `[100, 10, 1, 0.1]` (condition number 1000)
- 6 optimizer variants
- final-loss comparison after a fixed number of updates

## Pair-level scientific question

Can a **small Lanczos Ritz subspace** recover a meaningful fraction of the advantage of exact-Hessian democratic equalization, while staying much cheaper than the exact oracle reference?


## Truthfulness and interpretation contract

This notebook intentionally limits its claims to what the current implementation actually measures.

1. **Lanczos language:** the `Muon2_L10` and `Muon2_L5` methods use a **k-step Lanczos Ritz subspace**. They do **not** explicitly select “top + bottom eigenvectors” unless the implementation is changed to do so.
2. **Primary metric:** terminal loss after the configured number of optimization steps.
3. **Recovery metric:** percentage of the per-seed final-loss gap between SGD and Full Democratic that a method closes.
4. **Cost metric:** the current script reports a **counted matmul proxy**, not full FLOPs and not wall-clock truth.
5. **Oracle caveat:** `FullDemocratic` uses an exact finite-difference Hessian, but the counted proxy **excludes** that Hessian-construction overhead and also excludes decomposition costs such as eig/SVD/QR.

Accordingly, the cost/Pareto analysis below is useful for this toy implementation, but should be read as a **proxy-level comparison** rather than a hardware- or FLOP-faithful compute study.


In [ ]:
import importlib.util
import os
import platform
import sys
import time
from pathlib import Path

import matplotlib

try:
    from IPython import get_ipython
except Exception:
    get_ipython = lambda: None

if get_ipython() is None:
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    target_rel = Path('experiments/Tier_1_Core_Mechanism_Tests/MUON_2_0_hessian_adaptive/muon_2_0_hessian_adaptive.py')
    for candidate in [start, *start.parents]:
        if (candidate / target_rel).exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from current working directory.')


REPO_ROOT = find_repo_root()
EXP_DIR = REPO_ROOT / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'MUON_2_0_hessian_adaptive'
SCRIPT_PATH = EXP_DIR / 'muon_2_0_hessian_adaptive.py'

spec = importlib.util.spec_from_file_location('muon_2_0_hessian_adaptive_module', SCRIPT_PATH)
exp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp)

print('Repository root :', REPO_ROOT)
print('Experiment dir  :', EXP_DIR)
print('Script path     :', SCRIPT_PATH)
print('Notebook cwd    :', Path.cwd().resolve())
print('Python version  :', sys.version.split()[0])
print('Platform        :', platform.platform())
print('NumPy version   :', np.__version__)
print('Pandas version  :', pd.__version__)
print('Imported experiment module successfully without auto-running the experiment.')


## Reproducibility and runtime configuration

The default notebook configuration matches the script defaults. For automated smoke execution, the notebook also supports a reduced runtime mode via the environment variable:

- `MUON_HESSIAN_NOTEBOOK_SMOKE=1`

Smoke mode keeps the same methods and logic but reduces seeds, steps, and the LR grid.


In [ ]:
DEFAULT_CONFIG = exp.get_default_config()
SMOKE_MODE = os.environ.get('MUON_HESSIAN_NOTEBOOK_SMOKE', '0') == '1'
RUN_CONFIG = dict(DEFAULT_CONFIG)

if SMOKE_MODE:
    RUN_CONFIG.update({
        'n_steps': 20,
        'n_seeds': 1,
        'hessian_recompute_every': 10,
        'lr_candidates': [0.001, 0.01],
    })

print('Experiment name          :', exp.EXPERIMENT_NAME)
print('Counterpart notebook path:', exp.COUNTERPART_NOTEBOOK_PATH)
print('Methods                  :', RUN_CONFIG['methods'])
print('Smoke mode               :', SMOKE_MODE)
print('Config used in this run:')
display(pd.Series(RUN_CONFIG, name='value').to_frame())

print('Cost proxy caveat:')
print(' ', exp.COST_PROXY_CAVEAT)
print('Trajectory semantics:')
print(' ', exp.TRAJECTORY_SEMANTICS)


## Main experiment execution

This cell executes the **same reusable runner** used by the script. The returned `results` dictionary contains:
- configuration and seed list
- per-seed best LRs
- per-seed final losses
- per-seed loss trajectories
- counted matmul proxy totals
- aggregate recovery/Pareto summaries
- threshold-test summaries and verdict text


In [ ]:
t0 = time.time()
results = exp.run_experiment(config=RUN_CONFIG, verbose=True)
elapsed = time.time() - t0
print(f'Notebook run completed in {elapsed:.2f}s')


## Compact script-aligned report

The printed report below comes from the script's reporting helper, so the notebook and script remain aligned on the basic numerical summaries.


In [ ]:
exp.print_report(results)


## Summary table of methods

The main comparison table reports the toy-setting headline metrics:
- mean final loss across seeds
- spread across seeds
- counted matmul proxy total
- cost ratio relative to SGD under that proxy
- mean recovery percentage relative to the SGD-to-FullDemocratic gap
- proxy Pareto score = recovery / relative proxy cost


In [ ]:
summary_df = pd.DataFrame(exp.get_summary_rows(results))
summary_df = summary_df[[
    'display_name',
    'mean_final_loss',
    'std_final_loss',
    'mean_matmul_proxy',
    'cost_ratio_vs_sgd',
    'mean_recovery_pct',
    'std_recovery_pct',
    'pareto_score',
]]
summary_df = summary_df.rename(columns={
    'display_name': 'Method',
    'mean_final_loss': 'Mean final loss',
    'std_final_loss': 'Std final loss',
    'mean_matmul_proxy': 'Mean counted matmul proxy',
    'cost_ratio_vs_sgd': 'Cost ratio vs SGD',
    'mean_recovery_pct': 'Mean recovery %',
    'std_recovery_pct': 'Std recovery %',
    'pareto_score': 'Proxy Pareto score',
})
summary_df = summary_df.sort_values('Mean final loss').reset_index(drop=True)
display(summary_df.round(4))


## What the current cost proxy counts and excludes

This section makes the cost model explicit rather than implied.


In [ ]:
cost_note_df = pd.DataFrame([
    {
        'Method': exp.DISPLAY_NAMES[m],
        'Mean counted matmul proxy': results['aggregate']['method_stats'][m]['mean_matmul_proxy'],
        'Cost ratio vs SGD': results['aggregate']['method_stats'][m]['cost_ratio'],
        'Proxy interpretation': exp.METHOD_PROXY_NOTES[m],
    }
    for m in results['methods']
])
print(exp.COST_PROXY_CAVEAT)
display(cost_note_df.round({'Mean counted matmul proxy': 1, 'Cost ratio vs SGD': 3}))


## Mean loss trajectories across seeds

The script now records trajectories with **terminal-loss-aligned semantics**:
- `loss_curve[0]` is the initial loss before updates
- `loss_curve[t]` for `t >= 1` is the post-update loss after `t` optimization steps

The plot below shows mean ± 1 standard deviation across seeds.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
steps = np.array(results['aggregate']['trajectory_steps'])
for method in results['methods']:
    stats = results['aggregate']['method_stats'][method]
    mean_curve = np.array(stats['loss_curve_mean'])
    std_curve = np.array(stats['loss_curve_std'])
    ax.plot(steps, mean_curve, label=exp.DISPLAY_NAMES[method], linewidth=2)
    ax.fill_between(steps, np.maximum(mean_curve - std_curve, 1e-16), mean_curve + std_curve, alpha=0.18)
ax.set_yscale('log')
ax.set_xlabel('Optimization step')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Mean ± std loss trajectories across seeds')
ax.legend(loc='best', fontsize=9)
plt.show()


## Recovery vs proxy cost

This is the most honest version of the current efficiency comparison:
- x-axis: counted cost ratio vs SGD
- y-axis: mean recovery percentage
- special caveat: `FullDemocratic` is still plotted, but its exact-Hessian overhead is **under-counted**, so it should be interpreted as an oracle reference rather than a deployable efficiency point


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
frontier = results['aggregate']['pareto_frontier']
for method in results['methods']:
    stats = results['aggregate']['method_stats'][method]
    x = stats['cost_ratio']
    y = stats['mean_recovery']
    marker = 's' if method == 'FullDemocratic' else 'o'
    size = 120 if method in frontier else 80
    alpha = 0.9 if method in frontier else 0.75
    ax.scatter(x, y, s=size, marker=marker, alpha=alpha)
    ax.annotate(method, (x, y), textcoords='offset points', xytext=(6, 6), fontsize=9)

frontier_xy = [
    (
        results['aggregate']['method_stats'][m]['cost_ratio'],
        results['aggregate']['method_stats'][m]['mean_recovery'],
    )
    for m in frontier
]
frontier_xy = sorted(frontier_xy, key=lambda z: z[0])
ax.plot([x for x, _ in frontier_xy], [y for _, y in frontier_xy], linestyle='--', linewidth=1.5, color='black', alpha=0.7)
ax.axhline(100.0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('Counted cost ratio vs SGD')
ax.set_ylabel('Mean recovery (%)')
ax.set_title('Recovery vs counted matmul proxy cost')
plt.show()

print('Proxy Pareto frontier:', [exp.DISPLAY_NAMES[m] for m in frontier])


## Per-seed recovery comparison

The plot below checks whether observed gains are consistent across seeds or are dominated by a small subset of initializations.


In [ ]:
recovery_records = []
for method in results['methods']:
    vals = results['aggregate']['method_stats'][method]['recoveries']
    for seed, rec in zip(results['seeds'], vals):
        recovery_records.append({
            'Seed': seed,
            'Method': exp.DISPLAY_NAMES[method],
            'Recovery %': rec,
        })
recovery_df = pd.DataFrame(recovery_records)
display(recovery_df.pivot(index='Seed', columns='Method', values='Recovery %').round(2))

fig, ax = plt.subplots(figsize=(10, 6))
for method in results['methods']:
    sub = recovery_df[recovery_df['Method'] == exp.DISPLAY_NAMES[method]].sort_values('Seed')
    ax.plot(sub['Seed'], sub['Recovery %'], marker='o', linewidth=1.5, label=exp.DISPLAY_NAMES[method])
ax.axhline(100.0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('Seed')
ax.set_ylabel('Recovery (%)')
ax.set_title('Per-seed recovery relative to the SGD-to-FullDemocratic gap')
ax.legend(loc='best', fontsize=8)
plt.show()


## Threshold-test summary and calibrated verdict

These tests are simple descriptive thresholds, not formal statistical hypothesis tests. They are included because they are part of the script's intended reporting contract, but they should be read with toy-setting caution.


In [ ]:
threshold_df = pd.DataFrame(exp.get_threshold_rows(results))
threshold_df = threshold_df.rename(columns={
    'test_id': 'Test',
    'question': 'Question',
    'metric': 'Metric',
    'reference': 'Reference',
    'pass': 'Pass',
})
display(threshold_df)

verdict = results['aggregate']['verdict']
print('Verdict category :', verdict['category'])
print('Tests passed     :', f"{verdict['tests_passed']}/{verdict['tests_considered']}")
print('Summary          :', verdict['summary'])
print('Caveat           :', verdict['caveat'])
print('Best proxy Pareto:', exp.DISPLAY_NAMES[verdict['best_proxy_pareto_method']])


## Appendix A: local HVP and Lanczos sanity checks

These diagnostics are **not** the main evidence path. They help verify that the Hessian-oriented machinery is numerically coherent at a representative point.


In [ ]:
test_problem = exp.make_seed_problem(results['seeds'][0], config=RUN_CONFIG)
theta_test = test_problem['theta0']
target_test = test_problem['target']

H_test = exp.hessian_fn(theta_test, target_test)
rng = np.random.RandomState(0)
v = rng.randn(exp.N_PARAMS)
v /= np.linalg.norm(v)
Hv_exact = H_test @ v
Hv_fd = exp.hvp(theta_test, target_test, v, count=False)
rel_hvp_err = np.linalg.norm(Hv_exact - Hv_fd) / (np.linalg.norm(Hv_exact) + 1e-12)

ritz_vals_10, ritz_vecs_10 = exp.lanczos(theta_test, target_test, 10, count=False)
exact_evals = np.linalg.eigvalsh(H_test)

print('Relative HVP error vs exact Hessian product :', rel_hvp_err)
print('Exact Hessian eigenvalue range              :', (exact_evals.min(), exact_evals.max()))
print('Lanczos Ritz value range (k=10)             :', (ritz_vals_10.min(), ritz_vals_10.max()))
print('Ritz subspace shape                         :', ritz_vecs_10.shape)


## Appendix B: one-point direction comparison

At one representative point, compare the cosine similarity of several update directions with the exact democratic direction. This is only a local diagnostic, not a performance metric.


In [ ]:
g = exp.grad_fn(theta_test, target_test, count=False)
gmats = exp.grad_matrices(theta_test, target_test, count=False)
H_eigvals, H_eigvecs = np.linalg.eigh(H_test)

d_dem = exp.match_reference_norm(exp.democratic_direction(g, H_eigvecs), g)
d_muon = exp.muon_direction(theta_test, target_test, gmats=gmats, count=False)
d_muon = exp.match_reference_norm(d_muon, g)
d_l10 = exp.match_reference_norm(exp.muon2_lanczos_direction(g, ritz_vecs_10), g)

def cosine(a, b):
    return float(a @ b / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-12))

local_dir_df = pd.DataFrame([
    {'Direction': 'Gradient (SGD)', 'Cosine with democratic direction': cosine(g, d_dem)},
    {'Direction': 'Muon', 'Cosine with democratic direction': cosine(d_muon, d_dem)},
    {'Direction': 'Muon2_L10 Ritz', 'Cosine with democratic direction': cosine(d_l10, d_dem)},
    {'Direction': 'Full Democratic', 'Cosine with democratic direction': cosine(d_dem, d_dem)},
])
display(local_dir_df.round(4))


## Final conclusion for this first completion pass

This notebook now stays aligned with the script by **importing the reusable experiment runner** and presenting the results with explicit caveats. Any substantive scientific conclusion should be stated narrowly:

- only for this **4x4x3 deep-linear toy problem**,
- only for the current **six-method setup**,
- only under the current **counted-matmul proxy**,
- and without treating the proxy Pareto numbers as full compute truth.


In [ ]:
final_summary = pd.DataFrame(summary_df)
best_loss_method = final_summary.iloc[0]['Method']
verdict = results['aggregate']['verdict']

print('Best mean final-loss method in this run :', best_loss_method)
print('Proxy Pareto leader                    :', exp.DISPLAY_NAMES[verdict['best_proxy_pareto_method']])
print('Script-aligned verdict                 :', verdict['summary'])
print('Interpretation caveat                  :', verdict['caveat'])
